# 초구 예측 — model_07 (2단계: 구종에 B 중 앞 타석 초구 결과)

- 0장 = 모델 전 확인: C 투수 이력(0-1)·D 상대(0-2)·B 오늘 경기 흐름(0-3)이 model_06 위에 남긴 효과 (E는 보류)
  - C·D는 남은 효과 없음 (README 4-11·4-12), B는 앞 타석 초구 결과만 두 방법 모두 + 쪽 (+0.4·+0.7 ×1000)
- 모델: model_06에 **앞 타석 초구 결과**(바로 앞 타석 초구가 볼·루킹·헛스윙·파울·인플레이·없음) 변수 하나 묶음만 더함
  - B4·위치는 그대로, LightGBM 파라미터만 다시 고름
- 비교 대상: model_06 -> **채택** (2020 구종 −0.0003)


In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_07"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. C 투수 이력: model_06 위에 남은 효과

- 이유: model_05 0-2 사전 확인은 옛 B4 위에서 잼 (C 단독 0.6, A 위 −0.9, 전부에서 빼면 2.7) -> model_06에서 B4에 시즌 초 배수가 들어가 C(직전 경기 배합·역할 전환)와 겹칠 수 있음 -> 다시 잼
- C 변수 `fc.pitcher_history_features` (fp_common, model_07부터): 미래 정보 검사 통과 후 사용
  - 등판 간격 `rest`, 역할 전환 `role_change` (직전 등판과 선발/불펜 다름): 경기 기록
  - 모든 카운트 구종 배합 `mix_all_*` (전날까지, 반감 200구), 직전 날짜 경기 배합 `mix_last_*`: 트래킹 (전날까지)
  - 배합은 **배합 − 지금 B4 확률**로 넣음 (트리는 출발점 확률을 직접 못 보므로 "출발점과 얼마나 다른가"를 줌)
- 방법 (model_05 0-2와 같음): model_06 B4 log를 출발점으로 LightGBM, 학습 2017–18 (경기 20%는 조기 종료용), 확인 2019, 파라미터는 model_06 값 고정
  - A만(= model_06 구조) / A + C / A + C에서 하나씩 빼기 / A + 하나씩 -> A만 대비 2019 log loss 이득 × 1000 (양수 = 좋아짐), 경기 단위 부트스트랩

In [2]:
import lightgbm as lgb
from scipy.special import log_softmax

B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)  # model_06 B4 (기준점 지난 시즌 전부)

def type_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)
for yr in range(2017, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = type_online(fp[k], yr, **B4_P)
print("2019 B4 log loss (model_06 2-3과 같아야 함: 1.0937):", round(fc.prob_metrics(P_B4[val], y_type[val])["logloss"], 4))

2019 B4 log loss (model_06 2-3과 같아야 함: 1.0937): 1.0937


In [3]:
X_A = fc.situation_features(m, fp)  # A 이번 타석 상황 (model_05부터)
H = fc.pitcher_history_features(m, fp)  # C 투수 이력 (원래 값)
bad = fc.check_no_future(fc.pitcher_history_features, m, fp[fp["GYEAR"].ge(2017)], n=150)  # 미래 정보 검사
assert bad.empty, bad
X_C = pd.DataFrame({"rest": H["rest"], "role_change": H["role_change"],
                    **{"d_all_" + c: H["mix_all_" + c] - P_B4[:, i] for i, c in enumerate(CLASSES)},  # 모든 카운트 배합 − B4
                    **{"d_last_" + c: H["mix_last_" + c] - P_B4[:, i] for i, c in enumerate(CLASSES)}}, index=fp.index)  # 직전 경기 배합 − B4
X = pd.concat([X_A, X_C], axis=1)
display(X_C.describe().T[["count", "mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개


,count,mean,min,max
rest,50223.0,11.154,1.000,604.000
role_change,50223.0,0.066,0.000,1.000
d_all_CHUP,50223.0,0.004,-0.298,0.320
d_all_CURV,50223.0,-0.009,-0.312,0.207
d_all_CUTT,50223.0,-0.000,-0.217,0.187
d_all_FAST,50223.0,-0.012,-0.385,0.262
d_all_FORK,50223.0,0.026,-0.270,0.540
d_all_KNUC,50223.0,0.000,-0.029,0.063
d_all_SINK,50223.0,0.000,-0.039,0.126
d_all_SLID,50223.0,-0.002,-0.325,0.333


In [4]:
C_GROUPS = {"등판 간격": ["rest"], "역할 전환": ["role_change"],
            "모든 카운트 배합": ["d_all_" + c for c in CLASSES], "직전 경기 배합": ["d_last_" + c for c in CLASSES]}
C_COLS = sum(C_GROUPS.values(), [])
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)
GID = fp["gameID"].to_numpy()
itr = np.flatnonzero(fp["GYEAR"].between(2017, 2018).to_numpy())
i19 = np.flatnonzero(val)
gtr = np.unique(GID[itr])
es = np.isin(GID[itr], np.random.default_rng(1).choice(gtr, len(gtr) // 5, replace=False))  # 조기 종료용 경기 20%
tr, va = itr[~es], itr[es]
PARAMS = dict(objective="multiclass", num_class=K, learning_rate=0.03, num_leaves=7, min_data_in_leaf=200, lambda_l2=1000.0,
              feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True,
              force_row_wise=True, num_threads=4)  # model_06 값
LL_B4 = -np.log(np.clip(P_B4[i19, y_all[i19]], 1e-12, None))

def fit_ll(cols):  # 2019 공별 log loss, 트리 수
    dtr = lgb.Dataset(X[cols].iloc[tr], y_all[tr], init_score=INIT[tr], free_raw_data=False)
    dva = lgb.Dataset(X[cols].iloc[va], y_all[va], init_score=INIT[va], reference=dtr)
    bst = lgb.train(PARAMS, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(X[cols].iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INIT[i19]
    return -log_softmax(raw, axis=1)[np.arange(len(i19)), y_all[i19]], bst.best_iteration

def game_ci(d, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": GID[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

A = fc.A_COLS
runs = {"A만 (= model_06 구조)": A, "A + C 전부": A + C_COLS,
        **{f"A + C − {g}": A + [c for c in C_COLS if c not in cs] for g, cs in C_GROUPS.items()},
        **{f"A + {g}": A + cs for g, cs in C_GROUPS.items()}}
LL, rows = {}, []
for nm, cols in runs.items():
    LL[nm], it = fit_ll(cols)
for nm in runs:
    d = LL["A만 (= model_06 구조)"] - LL[nm]  # 양수 = A만보다 좋아짐
    lo, hi = game_ci(d)
    rows.append({"조합": nm, "트리 수": None, "2019 log loss": LL[nm].mean(), "A만 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
print(f"2019 B4 {LL_B4.mean():.4f}")
display(pd.DataFrame(rows).drop(columns="트리 수").set_index("조합").round(4))
rows = []
for g in C_GROUPS:  # 전부에서 빼면 잃는 몫 (순서와 무관한 고유 몫)
    d = LL[f"A + C − {g}"] - LL["A + C 전부"]
    lo, hi = game_ci(d)
    rows.append({"C 안 변수": g, "빼면 잃는 몫 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index("C 안 변수").round(2))

2019 B4 1.0937


,2019 log loss,A만 대비 이득 ×1000,95% 하한,95% 상한
조합,,,,
A만 (= model_06 구조),1.0637,0.0000,0.0000,0.0000
A + C 전부,1.0692,-5.4750,-8.7733,-1.9565
A + C − 등판 간격,1.0689,-5.2217,-8.5853,-1.8240
A + C − 역할 전환,1.0695,-5.7692,-9.1739,-2.3174
A + C − 모든 카운트 배합,1.0685,-4.7559,-7.8299,-1.7305
A + C − 직전 경기 배합,1.0693,-5.5416,-8.7887,-2.3748
A + 등판 간격,1.0649,-1.2256,-1.9557,-0.5185
A + 역할 전환,1.0639,-0.2278,-0.4849,0.0266
A + 모든 카운트 배합,1.0687,-4.9729,-8.1208,-1.9794


,빼면 잃는 몫 ×1000,95% 하한,95% 상한
C 안 변수,,,
등판 간격,-0.25,-0.64,0.11
역할 전환,0.29,0.05,0.54
모든 카운트 배합,-0.72,-2.52,1.02
직전 경기 배합,0.07,-1.77,1.89


#### 0-1-2. 2017 출발점 문제 확인

- 위에서 C가 A 위에 크게 나빠짐 (−5.5) -> 추정 원인: 2017은 지난 기록이 없어 B4가 그 시즌 리그 비율로 출발 -> 학습 해에는 "배합이 B4와 다르면 배합을 믿어라"가 맞고 2019에는 틀림
- 세 가지로 확인 (각 방법 안에서 A만 vs A + C, 파라미터는 위와 같음)
  1. 2018만 학습 (조기 종료용 경기 20%도 2018에서)
  2. 2017–18 학습, 전날까지 그 투수 초구 기록 100개 미만인 공은 학습에서 뺌 (2019 확인은 그대로 전부)
  3. 2017–18 학습 + B4 기록량 변수 `b4_n` (B4의 투수 유효 기록 수, 시즌 초 배수 반영) -> 트리가 "기록 적을 때만 배합을 믿어라"를 배울 수 있게


In [5]:
_, N_b4 = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, B4_P["half_life"], season_f=B4_P["season_f"]),
                    fp, ["pitcher_code"], CLASSES, season_f=B4_P["season_f"])
X["b4_n"] = N_b4[:, 0]  # B4 투수 유효 기록 수 (원-핫이라 9개 열 N이 같음 -> 첫 열)
_, N_raw = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES[:1]), fp, ["pitcher_code"], CLASSES[:1])
n_prior = N_raw[:, 0]  # 전날까지 그 투수 초구 수 (가중 없음)

def split(train_idx):  # 학습 행 -> (학습, 조기 종료용) 경기 20%
    g = np.unique(GID[train_idx])
    e = np.isin(GID[train_idx], np.random.default_rng(1).choice(g, len(g) // 5, replace=False))
    return train_idx[~e], train_idx[e]

def fit_ll2(cols, train_idx):  # fit_ll과 같고 학습 행만 바꿈
    t_, v_ = split(train_idx)
    dtr = lgb.Dataset(X[cols].iloc[t_], y_all[t_], init_score=INIT[t_], free_raw_data=False)
    dva = lgb.Dataset(X[cols].iloc[v_], y_all[v_], init_score=INIT[v_], reference=dtr)
    bst = lgb.train(PARAMS, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(X[cols].iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INIT[i19]
    return -log_softmax(raw, axis=1)[np.arange(len(i19)), y_all[i19]]

yr = fp["GYEAR"].to_numpy()
SETUPS = {"1. 2018만 학습": (np.flatnonzero(yr == 2018), []),
          "2. 2017–18, 기록 100개 미만 공 뺌": (np.flatnonzero((yr >= 2017) & (yr <= 2018) & (n_prior >= 100)), []),
          "3. 2017–18 + B4 기록량 변수": (itr, ["b4_n"])}
rows = []
for sname, (tidx, extra) in SETUPS.items():
    base = fit_ll2(A + extra, tidx)
    for nm, cs in {"A + C 전부": C_COLS, **{f"A + {g}": cs for g, cs in C_GROUPS.items()}}.items():
        d = base - fit_ll2(A + extra + cs, tidx)  # 양수 = 그 방법의 A만보다 좋아짐
        lo, hi = game_ci(d)
        rows.append({"방법": sname, "학습 초구": len(tidx), "A만 2019 log loss": base.mean(), "조합": nm,
                     "A만 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index(["방법", "조합"]).round(4))

학습 초구  A만 2019 log loss  A만 대비 이득 ×1000  95% 하한  95% 상한
방법                         조합                                                                    
1. 2018만 학습                A + C 전부       12395             1.069         -1.1827 -4.3814  2.0399
                           A + 등판 간격      12395             1.069         -0.8080 -1.3351 -0.2673
                           A + 역할 전환      12395             1.069          0.4806  0.2183  0.7340
                           A + 모든 카운트 배합  12395             1.069         -1.7722 -5.2368  1.5214
                           A + 직전 경기 배합   12395             1.069         -2.0416 -4.4653  0.4721
2. 2017–18, 기록 100개 미만 공 뺌 A + C 전부       20644             1.066         -1.3725 -4.2844  1.3502
                           A + 등판 간격      20644             1.066         -0.3369 -0.7763  0.0938
                           A + 역할 전환      20644             1.066         -0.0123 -0.2134  0.1801
                           A + 모든 카운트 배합  20644             1.066         -1.9618 -4.8613  0.8027
                           A + 직전 경기 배합   20644             1.066         -2.0045 -4.0303  0.1587
3. 2017–18 + B4 기록량 변수     A + C 전부       24636             1.065         -4.0268 -7.3350 -0.7409
                           A + 등판 간격      24636             1.065         -0.5685 -1.2565  0.1138
                           A + 역할 전환      24636             1.065          0.0742 -0.0888  0.2426
                           A + 모든 카운트 배합  24636             1.065         -3.4077 -6.6160 -0.4552
                           A + 직전 경기 배합   24636             1.065         -2.9068 -5.7537 -0.0317

### 🔍 0-1 결과
#### C 투수 이력은 model_06 위에 남은 효과 없음 -> 적용 안 함 / 크게 나빠진 건 대부분 2017 출발점 문제
- 원래 방법(2017–18 학습): A + C 전부 −5.5 (95% −8.8 ~ −2.0, 나빠짐), 배합 두 묶음이 −4.6 ~ −5.0, 등판 간격 −1.2, 역할 전환 −0.2
  - model_05 시점(옛 B4)에는 전부에서 빼면 2.7 -> 지금은 남은 몫 없음 (시즌 초 배수가 겹치는 몫을 가져감)
- 원인 확인 (0-1-2): 2017은 B4가 그 시즌 리그 비율로 출발 -> "배합이 B4와 다르면 배합을 믿어라"가 학습 해에만 맞음
  - 2018만 학습 −1.2 · 기록 100개 미만 공 뺌 −1.4 (둘 다 95% 구간에 0 포함) -> 나빠짐은 대부분 사라짐, 그래도 이득은 없음
  - 역할 전환만 2018 학습에서 +0.5 (작음), 다른 방법에서는 0
  - B4 기록량 변수(`b4_n`)를 줘도 −4.0 -> 트리가 "기록 적을 때만 믿기"를 못 배움
  - A 자체는 2017이 있어야 좋음 (A만 2019: 2017–18 1.0637, 2018만 1.069, 기록 적은 공 뺌 1.066) -> 학습에서 2017을 빼지 않음
- 다음 사전 확인부터 원래 방법 + "기록 100개 미만 공 뺌"을 같이 봄 (출발점·과거 기록과 비교하는 변수는 같은 2017 문제에 걸림)
- C 적용 안 함 (사용자 확인) -> 다음은 D 상대 (0-2)


### 0-2. D 상대: model_06 위에 남은 효과

- 이유: model_05 0-2 사전 확인 (옛 B4 위): D 단독 3.4·A 위 −2.1·전부에서 빼면 3.0 -> model_06 위에서 다시
- D 변수 (fp_common, model_07부터)
  - `fc.opponent_features`: 홈/원정, 타자 초구 스윙률 (전날까지 0-0 공, 그 공 전날까지 리그 스윙률 쪽으로 20구 당김), 타자 0-0 공 수 -> 미래 정보 검사
  - `fc.group_residual`: 타자별·상대 팀별 "지금 B4가 구종별로 빗나간 정도" (전날까지 (실제 원-핫 − B4) 합 / (공 수 + 50)) -> 표본 300개를 그 날짜 전 행만으로 직접 계산해 맞춰 봄
    - B4는 그 공 전날까지 정보로 만든 확률 (model_06 B4)
- 방법은 0-1과 같음 + 0-1-2 교훈대로 "기록 100개 미만 공 뺌"도 같이 (잔차는 B4와 비교하는 변수라 2017 출발점 문제에 걸릴 수 있음)


In [6]:
X_D = fc.opponent_features(m, fp)  # 홈/원정, 타자 스윙률, 타자 0-0 공 수
bad = fc.check_no_future(fc.opponent_features, m, fp[fp["GYEAR"].ge(2017)], n=150)
assert bad.empty, bad
E_bat = fc.group_residual(fp, P_B4, ["batter_code"], fp, k=50.0, prefix="enc_bat")  # 타자 잔차 (n, 9)
E_team = fc.group_residual(fp, P_B4, ["bat_team"], fp, k=50.0, prefix="enc_team")  # 상대 팀 잔차 (n, 9)
R = fp[CLASSES].to_numpy() - P_B4  # 직접 계산 확인: 표본 공마다 그 날짜 전 같은 타자 행만으로 합계
rng = np.random.default_rng(0)
miss = 0
for i in rng.choice(len(fp), 300, replace=False):
    msk = (fp["batter_code"].to_numpy() == fp["batter_code"].iat[i]) & (fp["game_date"].to_numpy() < fp["game_date"].iat[i])
    miss += not np.allclose(np.nansum(R[msk], 0) / (msk.sum() + 50.0), E_bat.iloc[i].to_numpy())
print(f"타자 잔차 직접 계산과 다른 공: {miss} / 300")
assert miss == 0
X = pd.concat([X.drop(columns=[c for c in X if c in X_D or c in E_bat or c in E_team]), X_D, E_bat, E_team], axis=1)
OPP_GROUPS = {"홈/원정": ["home"], "타자 스윙률": ["bat_swing", "bat_n"], "타자 잔차": list(E_bat), "상대 팀 잔차": list(E_team)}
OPP_COLS = sum(OPP_GROUPS.values(), [])
display(X[OPP_COLS].describe().T[["mean", "std", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
타자 잔차 직접 계산과 다른 공: 0 / 300


,mean,std,min,max
home,0.505,0.500,0.000,1.000
bat_swing,0.299,0.070,0.124,0.482
bat_n,150.562,126.159,0.000,555.000
enc_bat_CHUP,-0.001,0.021,-0.066,0.080
enc_bat_CURV,0.002,0.023,-0.066,0.085
enc_bat_CUTT,0.000,0.003,-0.027,0.030
enc_bat_FAST,-0.002,0.043,-0.155,0.161
enc_bat_FORK,0.001,0.013,-0.042,0.054
enc_bat_KNUC,0.000,0.001,-0.003,0.014
enc_bat_SINK,0.000,0.001,-0.002,0.013


In [7]:
rows = []
for sname, tidx in {"원래 (2017–18)": itr, "기록 100개 미만 공 뺌": SETUPS["2. 2017–18, 기록 100개 미만 공 뺌"][0]}.items():
    base = fit_ll2(A, tidx)
    runs_d = {"A + D 전부": A + OPP_COLS, **{f"A + {g}": A + cs for g, cs in OPP_GROUPS.items()}}
    if sname.startswith("원래"):
        runs_d |= {f"A + D − {g}": A + [c for c in OPP_COLS if c not in cs] for g, cs in OPP_GROUPS.items()}
    LLd = {nm: fit_ll2(cols, tidx) for nm, cols in runs_d.items()}
    for nm, ll_ in LLd.items():
        d = base - ll_  # 양수 = A만보다 좋아짐
        lo, hi = game_ci(d)
        rows.append({"방법": sname, "조합": nm, "A만 2019 log loss": base.mean(), "A만 대비 이득 ×1000": d.mean() * 1000,
                     "95% 하한": lo * 1000, "95% 상한": hi * 1000})
    if sname.startswith("원래"):
        for g in OPP_GROUPS:  # 전부에서 빼면 잃는 몫
            d = LLd[f"A + D − {g}"] - LLd["A + D 전부"]
            lo, hi = game_ci(d)
            rows.append({"방법": sname, "조합": f"(D 전부에서 {g} 빼면 잃는 몫)", "A만 2019 log loss": np.nan,
                         "A만 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index(["방법", "조합"]).round(4))

A만 2019 log loss  A만 대비 이득 ×1000  95% 하한  95% 상한
방법             조합                                                                        
원래 (2017–18)   A + D 전부                            1.0637         -2.9699 -5.1971 -0.8489
               A + 홈/원정                            1.0637         -0.2978 -0.9079  0.3061
               A + 타자 스윙률                          1.0637         -2.1342 -3.3367 -0.9244
               A + 타자 잔차                           1.0637         -1.9061 -3.9874  0.1717
               A + 상대 팀 잔차                         1.0637         -5.3185 -7.0613 -3.6139
               A + D − 홈/원정                        1.0637         -3.1598 -5.3653 -1.0232
               A + D − 타자 스윙률                      1.0637         -3.0903 -5.3910 -0.9408
               A + D − 타자 잔차                       1.0637         -4.2296 -6.1464 -2.4206
               A + D − 상대 팀 잔차                     1.0637         -2.1389 -4.2240 -0.0828
               (D 전부에서 홈/원정 빼면 잃는 몫)                  NaN          0.1899 -0.1933  0.6261
               (D 전부에서 타자 스윙률 빼면 잃는 몫)                NaN          0.1204 -0.3304  0.5757
               (D 전부에서 타자 잔차 빼면 잃는 몫)                 NaN          1.2597 -0.2914  2.8135
               (D 전부에서 상대 팀 잔차 빼면 잃는 몫)               NaN         -0.8310 -1.8658  0.1869
기록 100개 미만 공 뺌 A + D 전부                            1.0660         -1.8149 -3.6927  0.0382
               A + 홈/원정                            1.0660         -0.0510 -0.3942  0.2950
               A + 타자 스윙률                          1.0660         -2.2541 -3.3105 -1.2470
               A + 타자 잔차                           1.0660         -0.2844 -2.0657  1.4974
               A + 상대 팀 잔차                         1.0660         -3.1779 -4.8198 -1.4590

#### 0-2-2. 타자 스윙률만 (`bat_n` 뺌)

- 위에서 타자 스윙률 묶음이 두 방법 모두 나빠짐 -> 함께 넣은 타자 0-0 공 수 `bat_n`이 원인 후보: 기록이 해마다 쌓여 2019 값 범위가 학습 해보다 큼 (트리가 처음 보는 범위)
- `bat_n`을 빼고 스윙률만, D 전부에서도 `bat_n`만 뺌


In [8]:
yr_q = fp["GYEAR"].to_numpy()
display(pd.DataFrame({y: X.loc[yr_q == y, "bat_n"].describe()[["mean", "50%", "max"]] for y in (2017, 2018, 2019)}).round(1).rename_axis("bat_n"))
rows = []
for sname, tidx in {"원래 (2017–18)": itr, "기록 100개 미만 공 뺌": SETUPS["2. 2017–18, 기록 100개 미만 공 뺌"][0]}.items():
    base = fit_ll2(A, tidx)
    for nm, cols in {"A + 타자 스윙률 (bat_n 뺌)": A + ["bat_swing"], "A + bat_n만": A + ["bat_n"],
                     "A + D 전부 − bat_n": A + [c for c in OPP_COLS if c != "bat_n"]}.items():
        d = base - fit_ll2(cols, tidx)
        lo, hi = game_ci(d)
        rows.append({"방법": sname, "조합": nm, "A만 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index(["방법", "조합"]).round(2))

,2017,2018,2019
bat_n,,,
mean,42.4,120.0,185.6
50%,36.0,124.0,203.0
max,150.0,279.0,426.0


A만 대비 이득 ×1000  95% 하한  95% 상한
방법             조합                                                  
원래 (2017–18)   A + 타자 스윙률 (bat_n 뺌)           -0.91   -1.87    0.11
               A + bat_n만                     -2.93   -4.01   -1.87
               A + D 전부 − bat_n               -3.49   -5.60   -1.46
기록 100개 미만 공 뺌 A + 타자 스윙률 (bat_n 뺌)           -0.49   -1.28    0.35
               A + bat_n만                     -3.48   -4.32   -2.63
               A + D 전부 − bat_n               -1.55   -3.41    0.31

### 🔍 0-2 결과
#### D 상대는 model_06 위에 남은 효과 없음 -> 적용 안 함 / 타자 스윙률이 나빠진 건 함께 넣은 타자 기록 수 `bat_n` 때문
- A + D 전부: 원래 −3.0 (나빠짐), 기록 100개 미만 공 뺌 −1.8 (차이 없음) / D 전부에서 하나씩 빼도 고유 몫 없음 (타자 잔차 +1.3, 95% 구간에 0 포함)
- 홈/원정 −0.3 · −0.1, 타자 잔차 −1.9 · −0.3 (모두 차이 없음)
- 상대 팀 잔차 −5.3 · −3.2 (두 방법 모두 나빠짐): 팀 잔차는 사실상 그 팀을 상대한 투수들의 쏠림 -> 다음 해에 반복 안 됨 (model_05 0-2와 같음)
- 타자 스윙률 묶음 −2.1 · −2.3 (나빠짐) -> 0-2-2: `bat_n`만 −2.9 · −3.5 (나빠짐), 스윙률만 −0.9 · −0.5 (차이 없음)
  - `bat_n`은 기록이 해마다 쌓여 범위가 달라짐 (평균 2017 42 -> 2018 120 -> 2019 186) -> 트리가 학습 해에 없던 범위로 예측
  - 교훈: 데이터가 쌓일수록 커지는 변수(누적 기록 수 등)는 넣지 않음, 비율이나 반감기 준 값처럼 해마다 범위가 비슷한 형태로
- D 적용 안 함 (사용자 확인), model_06 그대로 -> 다음은 B 오늘 경기 흐름 (0-3), E는 보류


### 0-3. B 오늘 경기 흐름: model_06 위에 남은 효과

- 이유: model_05 0-2 사전 확인 (옛 B4 위): B 단독 11.1이지만 A 위 −0.6 (대부분 "앞 타석 없음 = 그 경기 첫 타자"라 A의 투구 수와 겹침), 전부에서 빼면 1.4 -> model_06 위에서 다시
- B 변수 `fc.game_flow_features` (fp_common, model_07부터, 같은 경기·같은 투수의 그 타석 앞 경기 기록): 미래 정보 검사
  - 앞 공 볼 비율 `ball_rate`, 앞 타석 초구 스트라이크(볼 아님) 비율 `fs_rate` (앞 공·타석 없으면 빈 값)
  - 바로 앞 타석 초구 결과 `prev_first_*` (없음·볼·루킹·헛스윙·파울·인플레이), 바로 앞 타석 결과 `prev_how_*` (없음·안타·홈런·볼넷·사구·삼진·아웃·기타) 원-핫
  - 비율·범주라 해마다 범위가 같음 (0-2 `bat_n` 문제 없음)
- 방법은 0-1·0-2와 같음 (원래 + 기록 100개 미만 공 뺌)


In [9]:
X_B = fc.game_flow_features(m, fp)
bad = fc.check_no_future(fc.game_flow_features, m, fp[fp["GYEAR"].ge(2017)], n=150)
assert bad.empty, bad
X = pd.concat([X.drop(columns=[c for c in X if c in X_B]), X_B], axis=1)
B_GROUPS = {"앞 공 볼 비율": ["ball_rate"], "앞 타석 초구 스트라이크 비율": ["fs_rate"],
            "앞 타석 초구 결과": [c for c in X_B if c.startswith("prev_first_")], "앞 타석 결과": [c for c in X_B if c.startswith("prev_how_")]}
B_COLS = sum(B_GROUPS.values(), [])
rows = []
for sname, tidx in {"원래 (2017–18)": itr, "기록 100개 미만 공 뺌": SETUPS["2. 2017–18, 기록 100개 미만 공 뺌"][0]}.items():
    base = fit_ll2(A, tidx)
    runs_b = {"A + B 전부": A + B_COLS, **{f"A + {g}": A + cs for g, cs in B_GROUPS.items()}}
    if sname.startswith("원래"):
        runs_b |= {f"A + B − {g}": A + [c for c in B_COLS if c not in cs] for g, cs in B_GROUPS.items()}
    LLb = {nm: fit_ll2(cols, tidx) for nm, cols in runs_b.items()}
    for nm, ll_ in LLb.items():
        d = base - ll_
        lo, hi = game_ci(d)
        rows.append({"방법": sname, "조합": nm, "A만 대비 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
    if sname.startswith("원래"):
        for g in B_GROUPS:  # 전부에서 빼면 잃는 몫
            d = LLb[f"A + B − {g}"] - LLb["A + B 전부"]
            lo, hi = game_ci(d)
            rows.append({"방법": sname, "조합": f"(B 전부에서 {g} 빼면 잃는 몫)", "A만 대비 이득 ×1000": d.mean() * 1000,
                         "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index(["방법", "조합"]).round(2))

미래 정보 검사: 150개 중 다른 공 0개


A만 대비 이득 ×1000  95% 하한  95% 상한
방법             조합                                                               
원래 (2017–18)   A + B 전부                                    -1.21   -2.55    0.08
               A + 앞 공 볼 비율                                -1.31   -2.37   -0.32
               A + 앞 타석 초구 스트라이크 비율                        -0.73   -1.73    0.16
               A + 앞 타석 초구 결과                               0.38   -0.30    1.01
               A + 앞 타석 결과                                  0.20   -0.39    0.77
               A + B − 앞 공 볼 비율                             0.01   -1.24    1.15
               A + B − 앞 타석 초구 스트라이크 비율                    -0.32   -1.63    0.81
               A + B − 앞 타석 초구 결과                          -1.73   -3.01   -0.55
               A + B − 앞 타석 결과                             -1.43   -2.73   -0.15
               (B 전부에서 앞 공 볼 비율 빼면 잃는 몫)                   -1.22   -2.06   -0.39
               (B 전부에서 앞 타석 초구 스트라이크 비율 빼면 잃는 몫)           -0.89   -1.56   -0.25
               (B 전부에서 앞 타석 초구 결과 빼면 잃는 몫)                  0.52   -0.01    1.03
               (B 전부에서 앞 타석 결과 빼면 잃는 몫)                     0.22   -0.21    0.63
기록 100개 미만 공 뺌 A + B 전부                                    -0.04   -1.17    1.01
               A + 앞 공 볼 비율                                -0.14   -1.05    0.68
               A + 앞 타석 초구 스트라이크 비율                        -0.16   -0.95    0.68
               A + 앞 타석 초구 결과                               0.72    0.29    1.17
               A + 앞 타석 결과                                  0.28   -0.19    0.74

### 🔍 0-3 결과
#### B 오늘 경기 흐름은 거의 다 A와 겹침 -> 앞 타석 초구 결과만 작게 남음 (model_07로 시험)
- A + B 전부: 원래 −1.2 · 기록 100개 미만 공 뺌 0.0 (둘 다 95% 구간에 0 포함)
- 앞 공 볼 비율 −1.3 (원래, 나빠짐)·앞 타석 초구 스트라이크 비율 −0.7 -> B 전부에서 빼면 오히려 +1.2 · +0.9 좋아짐
- **앞 타석 초구 결과**: +0.4 (원래, 차이 없음) · **+0.7** (기록 100개 미만 공 뺌, 95% +0.3 ~ +1.2), B 전부에서 빼면 0.5 잃음 -> 두 방법 모두 + 쪽인 유일한 변수
- 앞 타석 결과 +0.2 · +0.3 (차이 없음)
- model_05 0-2의 B 단독 몫(11.1)은 "그 경기 첫 타자"와 겹쳐 A가 이미 가져감 (다시 확인)
- -> 모델: A + 앞 타석 초구 결과 (사용자 선택)


## 모델

In [10]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종 B4: model_06 값 고정 (이번에 바꾸지 않음) / 위치: model_03 그대로
- 변수: A 이번 타석 상황 + **앞 타석 초구 결과** (0-3에서 유일하게 두 방법 모두 + 쪽)
- LightGBM: 변수가 바뀌어 연결된 파라미터만 다시 고름 (2-3)

In [11]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM (A + 앞 타석 초구 결과)

### 2-1. B4 (model_06과 같음)

In [12]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06 2-1 고른 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06 B4 (재현 확인)")
m06 = pd.read_parquet("results/model_06/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06 B4 (재현 확인)


### 2-2. 변수: A + 앞 타석 초구 결과

- `fc.situation_features` (A, model_05부터) + `fc.game_flow_features`의 `prev_first_*` (바로 앞 타석 초구 결과: 없음·볼·루킹·헛스윙·파울·인플레이)
- 미래 정보 검사: 두 함수 모두 (A는 model_05·06에서도 통과)

In [13]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-3. LightGBM 다시 고르기 (2019)

- 변수가 바뀌었으므로 연결된 LightGBM 파라미터만 다시 고름 (B4·learning_rate 등은 그대로)
- model_06 1등 주변 작은 격자 (잎 7·15 × 잎당 최소 20·50·200 × lambda_l2 300·1000·3000), model_06 값(7, 200, 1000) 포함 = 대조군
- 1등이 범위 끝이면 바깥으로 넓혀 다시 (lambda_l2 ×3, 잎 ×2, 최대 2번), 선택 규칙 model_06과 같음 (0.0002 안이면 잎 적게 -> 트리 적게)

In [14]:
# 격자 도구 (model_06에서 복사)
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

In [15]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05·06과 같음

def fit_type(train_years, lp, rounds=None, val_year=None):  # 학습 해로 LightGBM 학습 (rounds 없으면 val_year로 조기 종료)
    tr = np.flatnonzero(fp["GYEAR"].between(*train_years).to_numpy())
    dtr = lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr], free_raw_data=False)
    if rounds is None:
        va = np.flatnonzero(fp["GYEAR"].eq(val_year).to_numpy())
        dva = lgb.Dataset(X_all.iloc[va], y_all[va], init_score=INIT[va], reference=dtr)
        return lgb.train({**BASE, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE, **lp}, dtr, rounds)

def type_prob(bst, idx, rounds=None):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    raw = bst.predict(X_all.iloc[idx], raw_score=True, num_iteration=rounds or bst.best_iteration) + INIT[idx]
    return np.exp(log_softmax(raw, axis=1))

iv = np.flatnonzero(val)
def score_lgb(num_leaves, min_data_in_leaf, lambda_l2):  # LightGBM 한 조합의 2019 log loss, 트리 수
    lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
    bst = fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), lp, val_year=fc.VAL_YEAR)
    return {"트리 수": bst.best_iteration, "logloss": fc.prob_metrics(type_prob(bst, iv), y_type[val])["logloss"]}

GRID_LGB = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
res_lgb = grid_search(score_lgb, GRID_LGB)
for rnd in range(2):  # 1등이 범위 끝이면 넓힘
    b = res_lgb.iloc[0]
    add = {}
    if b.lambda_l2 == res_lgb.lambda_l2.max():
        add["lambda_l2"] = [b.lambda_l2 * 3]
    if b.num_leaves == res_lgb.num_leaves.max():
        add["num_leaves"] = [int(b.num_leaves * 2)]
    if not add:
        break
    cand = {"num_leaves": [int(b.num_leaves)], "min_data_in_leaf": [int(b.min_data_in_leaf)], "lambda_l2": [float(b.lambda_l2)], **add}
    print(f"{rnd + 1}번째 넓히기:", add)
    res_lgb = pd.concat([res_lgb, grid_search(score_lgb, cand)]).drop_duplicates(list(GRID_LGB)).sort_values("logloss").reset_index(drop=True)
TOL = 2e-4
ok = res_lgb[res_lgb["logloss"] <= res_lgb["logloss"].iloc[0] + TOL]
pick = ok.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 덜 극단적인 값: 잎 적게 -> 트리 적게
LGB_P = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
LGB_ROUNDS = int(pick["트리 수"])
ctl = res_lgb[(res_lgb.num_leaves == 7) & (res_lgb.min_data_in_leaf == 200) & (res_lgb.lambda_l2 == 1000)].iloc[0]  # model_06 값
b4_ll = fc.prob_metrics(P_B4[iv], y_type[val])["logloss"]
print(f"B4 2019 log loss {b4_ll:.4f} / LightGBM 1등 {res_lgb.logloss.iloc[0]:.4f} / 대조군(model_06 값) {ctl.logloss:.4f} / "
      f"선택 {pick.logloss:.4f} {LGB_P}, 트리 {LGB_ROUNDS}개")
display(res_lgb.head(10).round(5))

1번째 넓히기: {'lambda_l2': [np.float64(9000.0)]}
B4 2019 log loss 1.0937 / LightGBM 1등 1.0619 / 대조군(model_06 값) 1.0621 / 선택 1.0621 {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0}, 트리 730개


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,7,20,3000.0,1794,1.06195
1,7,50,3000.0,1885,1.06196
2,7,20,9000.0,4820,1.06197
3,15,50,3000.0,1460,1.06201
4,7,200,3000.0,1726,1.06202
5,15,20,3000.0,1493,1.06203
6,15,200,3000.0,1549,1.06207
7,15,200,1000.0,574,1.06207
8,15,50,1000.0,573,1.06208
9,15,20,1000.0,572,1.06210


### 🥇1등🥇 LightGBM
#### model_06 값과 거의 같은 곳: 잎 7, 잎당 최소 20, lambda_l2 1000, 트리 730개 (2019 1.0621)
- 1등(잎 7·잎당 20·λ 3000, 트리 1,794개) 1.06195 vs 고른 값 1.0621 -> 0.0002 안이라 평평한 구간 규칙으로 트리 적은 조합
- λ 1등이 범위 끝(3000) -> 9000으로 넓혀 확인 (1.06197, 트리 4,820개), 나아지지 않음
- 대조군(model_06 값 잎 7·잎당 200·λ 1000) 1.0621 -> 사실상 같음


### 2-4. 최종 학습과 구종 확률

- 2019 예측: 2017–18 학습 모델 / 2020 예측: 같은 파라미터·트리 수로 2017–19 다시 학습

In [16]:
BST = {fc.VAL_YEAR: fit_type((TRAIN_FIRST, fc.VAL_YEAR - 1), LGB_P, LGB_ROUNDS),
       fc.TEST_YEAR: fit_type((TRAIN_FIRST, fc.TEST_YEAR - 1), LGB_P, LGB_ROUNDS)}
P_TYPE_ALL = np.full((len(fp), K), np.nan)
for yr, bst in BST.items():
    idx = np.flatnonzero(fp["GYEAR"].eq(yr).to_numpy())
    P_TYPE_ALL[idx] = type_prob(bst, idx, LGB_ROUNDS)
P10 = {yr: fc.align(m06, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_06 구종 확률 (fp 그 해 순서)
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_06", P10[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
imp = pd.DataFrame({yr: pd.Series(b.feature_importance("gain"), index=FEAT) for yr, b in BST.items()})
display((imp / imp.sum()).sort_values(fc.VAL_YEAR, ascending=False).round(3).rename_axis("변수 (gain 비율)"))

log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_06    1.0624  0.5360  0.8163
     model_07    1.0621  0.5380  0.8174
2020 B4          1.0916  0.5264  0.8179
     model_06    1.0574  0.5330  0.8203
     model_07    1.0570  0.5340  0.8209

,2019,2020
변수 (gain 비율),,
pc,0.270,0.265
out_cn,0.119,0.125
sdiff,0.109,0.105
batorder,0.095,0.085
inning,0.094,0.091
base2,0.073,0.088
base1,0.071,0.064
tto,0.060,0.065
base3,0.041,0.047


### 2-5. 진단 (2019)

- 앞 타석 초구 결과별 구종 log loss (model_06 vs model_07) -> 어느 결과 뒤에서 이득이 나는지
- 상황별 직구 치우침·포함률 gap, 구종 1등 보정 구간표: model_06 2-5와 같은 표

In [17]:
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
pf = X_all.loc[val, ["prev_first_" + c for c in fc.FIRST_CATS]].to_numpy().argmax(1)
dv = q_val.assign(m06=nll(P10[fc.VAL_YEAR], y_type[val]), m07=nll(P_TYPE_ALL[val], y_type[val]), 앞타석초구=np.array(fc.FIRST_CATS)[pf])
t = dv.groupby("앞타석초구")[["m06", "m07"]].mean().reindex(fc.FIRST_CATS)
t.insert(0, "초구", dv.groupby("앞타석초구").size())
t["차이"] = t["m07"] - t["m06"]
yy = y_type[val]
for nm, P in [("m06", P10[fc.VAL_YEAR]), ("m07", P_TYPE_ALL[val])]:  # 직구 실제 − 예측
    t["직구 실제−예측 " + nm] = pd.Series((yy == fc.TYPE_IDX["FAST"]) - P[:, fc.TYPE_IDX["FAST"]]).groupby(np.array(fc.FIRST_CATS)[pf]).mean()
display(t.rename(columns={"m06": "model_06", "m07": MODEL}).round(4))

,초구,model_06,model_07,차이,직구 실제−예측 m10,직구 실제−예측 m11
앞타석초구,,,,,,
없음,1289,0.6573,0.6573,0.0000,0.0232,0.0237
볼,4811,1.1080,1.1085,0.0005,0.0001,0.0020
루킹,3721,1.1452,1.1458,0.0006,-0.0279,-0.0251
헛스윙,767,1.0469,1.0475,0.0006,0.0095,0.0163
파울,1349,1.0831,1.0832,0.0001,0.0014,0.0142
인플레이,1521,1.0479,1.0422,-0.0057,0.0402,0.0111


In [18]:
def ctx_check(P, msk):
    d = fp[msk].assign(**{c: X_all.loc[msk, c] for c in ["pc", "tto", "starter", "sdiff"]})  # X_all은 fp와 같은 순서
    y = y_type[msk]
    s = fc.show_types(P)
    i = np.arange(len(y))
    d["rFAST"] = (y == fc.TYPE_IDX["FAST"]) - P[:, fc.TYPE_IDX["FAST"]]
    d["gap"] = s[i, y] - (P * s).sum(1)
    cols = {"앞선 투구 수": pd.cut(d["pc"], [0, 1, 16, 31, 61, 999], right=False).astype(str),
            "같은 타자 몇 번째": d["tto"].clip(upper=3).astype(str), "아웃": d["out_cn"].clip(upper=2).astype(str),
            "득점권": np.where(d["base2"] + d["base3"] > 0, "득점권", "아님"), "선발/불펜": np.where(d["starter"] == 1, "선발", "불펜")}
    out = []
    for nm, c in cols.items():
        t = d.groupby(np.asarray(c))[["rFAST", "gap"]].mean()
        t.index = pd.MultiIndex.from_product([[nm], t.index], names=["변수", "값"])
        out.append(t)
    return pd.concat(out)

display(pd.concat({"model_06": ctx_check(P10[fc.VAL_YEAR], val), MODEL: ctx_check(P_TYPE_ALL[val], val)}, axis=1).round(3))
for nm, P in [("model_06", P10[fc.VAL_YEAR]), (MODEL, P_TYPE_ALL[val])]:
    print(nm, "2019 구종 1등 ECE", round(fc.calib_report(P, y_type[val], CLASSES)[0].loc["1등 예측", "ECE"], 4))

model_06        model_07       
                        rFAST    gap    rFAST    gap
변수         값                                        
앞선 투구 수    [0, 1)       0.024  0.037    0.024  0.036
           [1, 16)      0.003  0.014    0.003  0.016
           [16, 31)    -0.009 -0.007   -0.010 -0.009
           [31, 61)    -0.009 -0.021   -0.010 -0.021
           [61, 999)    0.003 -0.018    0.003 -0.020
같은 타자 몇 번째 1            0.006  0.012    0.006  0.013
           2           -0.010 -0.023   -0.010 -0.024
           3           -0.005 -0.019   -0.006 -0.021
아웃         0            0.015  0.007    0.015  0.007
           1           -0.008 -0.008   -0.008 -0.009
           2           -0.009 -0.007   -0.009 -0.007
득점권        득점권         -0.018 -0.013   -0.018 -0.014
           아님           0.006  0.001    0.006  0.001
선발/불펜      불펜          -0.001  0.013   -0.000  0.012
           선발          -0.000 -0.009   -0.000 -0.009

model_06 2019 구종 1등 ECE 0.0123
model_07 2019 구종 1등 ECE 0.0141


In [19]:
# 구종 1등 보정 구간표 (2019·2020): model_06 vs model_07
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("model_06", P10[yr]), (MODEL, P_TYPE_ALL[msk])]:
        rep, tabs = fc.calib_report(P, y_type[msk], CLASSES)
        t = tabs["1등 예측"]
        print(f"{yr} {nm}: 1등 ECE {rep.loc['1등 예측', 'ECE']:.4f}")
        display(t[t["공수"] >= 30].round(3).T)

2019 model_06: 1등 ECE 0.0123


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,91.000,1820.000,3876.000,3490.000,2269.000,1249.000,542.000,121.000
예측평균,0.285,0.361,0.452,0.546,0.646,0.744,0.846,0.924
실제비율,0.341,0.364,0.447,0.526,0.632,0.738,0.886,0.967
차이,0.056,0.003,-0.005,-0.020,-0.014,-0.006,0.040,0.043


2019 model_07: 1등 ECE 0.0141


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,94.000,1826.000,3854.000,3499.000,2253.000,1274.000,540.000,118.000
예측평균,0.285,0.361,0.453,0.546,0.645,0.744,0.846,0.925
실제비율,0.330,0.377,0.445,0.529,0.630,0.737,0.889,0.966
차이,0.045,0.015,-0.007,-0.017,-0.015,-0.006,0.043,0.042


2020 model_06: 1등 ECE 0.0168


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,230.000,1792.000,3504.000,3075.000,2003.000,1207.000,683.000,121.000
예측평균,0.270,0.364,0.452,0.547,0.645,0.746,0.842,0.923
실제비율,0.296,0.358,0.442,0.518,0.628,0.742,0.881,0.975
차이,0.026,-0.006,-0.010,-0.029,-0.017,-0.003,0.040,0.052


2020 model_07: 1등 ECE 0.0142


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,232.000,1813.000,3484.000,3096.000,1956.000,1228.000,680.000,125.000
예측평균,0.271,0.363,0.452,0.547,0.645,0.745,0.842,0.923
실제비율,0.272,0.359,0.442,0.523,0.632,0.741,0.876,0.976
차이,0.001,-0.004,-0.010,-0.025,-0.013,-0.004,0.035,0.053


### 🔍 진단
#### 이득은 거의 다 "앞 타석 초구가 인플레이"였을 때
- 앞 타석 초구 결과별 구종 log loss (2019, model_07 − model_06): 인플레이 −0.0057 (1,521개), 없음 0, 볼·루킹·헛스윙·파울 +0.0001 ~ +0.0006
  - 인플레이 뒤 직구 실제 − 예측: model_06 +0.040 -> model_07 +0.011 -> 앞 타자가 초구를 쳤으면 다음 타자에게 직구가 더 많음 (model_06은 이를 몰랐음)
- 상황별 직구 치우침·포함률 gap: model_06과 같음
- 구종 1등 ECE: 2019 0.012 -> 0.014, 2020 0.017 -> 0.014 (방향이 해마다 다름, 구간 모양 그대로) -> 계속 기록만


## 3. ② 위치 (model_03과 같음)

In [21]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [22]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_06과 같음, 구종 확률만 2장 결과(`P_TYPE_ALL`)

In [23]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [24]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종과 무관한 열(실제 구종 기준 위치)은 model_06과 같아야 함
chk = pp.merge(m06, on=fc.KEYS, suffixes=("", "_03"))
same = ["ll_zone_true", "nll_coord_true", "dist_true"]
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_03"], equal_nan=True)]
assert len(chk) == len(pp) and not bad, f"model_06과 다른 열: {bad}"
print("실제 구종 기준 위치 = model_06 (재현 확인)")

(26073, 48)
실제 구종 기준 위치 = model_06 (재현 확인)


## 5. 평가

- 채점은 `fc.evaluate`, model_06과 나란히 / 경기 단위 부트스트랩 (model_06 − model_06, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [25]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m06_sum = eval_years(m06)
display(pd.concat({MODEL: summary, "model_06": m06_sum, "차이 (model_06 대비)": summary - m06_sum}, axis=1).round(4))

model_07          model_06          차이 (model_06 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0621   1.0570   1.0624   1.0574          -0.0003 -0.0003
구종 top1               0.5380   0.5340   0.5360   0.5330           0.0019  0.0010
구종 top2               0.8174   0.8209   0.8163   0.8203           0.0010  0.0006
구종 top3               0.9371   0.9413   0.9375   0.9409          -0.0004  0.0005
구종 macro F1           0.3205   0.3510   0.3192   0.3496           0.0013  0.0015
24칸 logloss (실전)      2.9495   2.9456   2.9494   2.9457           0.0001 -0.0001
24칸 top1              0.1298   0.1340   0.1301   0.1339          -0.0003  0.0002
24칸 top3              0.2893   0.2912   0.2897   0.2913          -0.0004 -0.0002
H/S/C logloss         1.0748   1.0767   1.0748   1.0768          -0.0000 -0.0000
H/S/C 정확도             0.3954   0.3891   0.3951   0.3883           0.0003  0.0009
24칸 logloss (실제 구종)   2.8933   2.8967   2.8933   2.8967           0.0000  0.0000
구종×칸 logloss          3.9554   3.9530   3.9556   3.9534          -0.0003 -0.0003
좌표 평균거리 cm           32.6877  31.8346  32.6861  31.8351           0.0016 -0.0005
좌표 NLL                9.3174   9.2662   9.3173   9.2663           0.0001 -0.0001

In [26]:
display(fc.bootstrap_diff(fc.pitch_scores(m06, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
# 실전 출력: 누적 80% 보여 주기 (2020)
for nm, P_ in [("model_06", m06), (MODEL, pp)]:
    d = fc.align(P_, fp, fc.TEST_YEAR)
    Pt_, yt_ = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    s = fc.show_types(Pt_)
    print(f"{nm}: 보여 준 구종 수 평균 {s.sum(1).mean():.2f}, 포함률 {s[np.arange(len(yt_)), yt_].mean():.3f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,-0.0003,-0.0010,0.0004,차이 불확실
ll_zone,-0.0001,-0.0002,0.0001,차이 불확실
ll_joint,-0.0003,-0.0010,0.0004,차이 불확실
nll_coord,-0.0001,-0.0003,0.0001,차이 불확실
dist,-0.0005,-0.0034,0.0026,차이 불확실


model_06: 보여 준 구종 수 평균 2.40, 포함률 0.904
model_07: 보여 준 구종 수 평균 2.40, 포함률 0.904


### 🥇1등🥇
#### 2020 구종 log loss 1.0574 -> 1.0570 (−0.0003), 2019도 −0.0003, 부트스트랩 6개 점수 모두 차이 불확실·나빠짐 없음 -> 채택
- 효과는 작지만 두 해 같은 방향, 규칙상 2019 1등 + 2020 나빠진 점수 없음 -> 채택 (효과가 작다고 빼면 2020으로 고르는 셈)
  - model_04(2019 차이 −0.00006, 채택 안 함)보다 5배 크고 두 해 같은 방향
- top1 +0.001, 24칸 실전 −0.0001, 누적 80% 보여 주기 그대로 (평균 2.40개, 포함률 90.4%)
- **채택 -> 다음 비교 출발점** (사용자 확인)


## 6. 저장

- `results/model_07/` -> `compare.ipynb` 실행하면 자동 포함

In [27]:
fc.save_results(MODEL, pp, {"summary": summary, "lgb_grid": res_lgb})

저장: results\model_07 ['lgb_grid.csv', 'per_pitch.parquet', 'summary.csv']
